# 05B Global Model Transfer Only

This notebook runs a strict saved-artifact transfer evaluation on M5 using synthetic-trained global boosting artifacts.

Important:
- this is an honest saved-model transfer test
- it does **not** retrain on M5
- it produces transfer metrics on monthly aggregated M5 data
- it is not the same as a Kaggle daily submission, because the saved synthetic artifacts are monthly models. This notebook is transfer evaluation only and does not generate uploadable XGBOOST/CATBOOST Kaggle CSVs.
- prerequisite: saved artifacts must already exist under `modeling/outputs/artifacts` (for example from notebook 02 global model training and artifact save)

In [27]:
from pathlib import Path
import pandas as pd

M5_DIR = Path('/Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/external-data/m5-forecasting-accuracy')
REPORTS_DIR = Path('/Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/modeling/outputs/reports')
SCRIPT = Path('/Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/modeling/scripts/m5_saved_artifact_transfer.py')
TAG = 'portable_m5_transfer'

def resolve_report_path(kind: str, preferred_tag: str = TAG) -> Path:
    if not REPORTS_DIR.exists():
        raise FileNotFoundError(
            f'Reports directory not found: {REPORTS_DIR}. Run the transfer cell above first.'
        )

    preferred = REPORTS_DIR / f'{preferred_tag}_{kind}.csv'
    if preferred.exists():
        return preferred

    available = sorted(p.name for p in REPORTS_DIR.glob(f'{preferred_tag}_*.csv'))
    raise FileNotFoundError(
        f'Missing transfer report: {preferred.name}. Run the transfer cell above first. '
        f'Available transfer CSVs for tag {preferred_tag!r}: {available}'
    )


## Run Transfer Evaluation

In [28]:
ARTIFACTS_DIR = Path('/Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/modeling/outputs/artifacts')
required_checks = [
    ARTIFACTS_DIR / 'P' / 'xgboost_h1' / 'production' / 'metadata.json',
    ARTIFACTS_DIR / 'P' / 'catboost_h1' / 'production' / 'metadata.json',
]
missing = [str(p) for p in required_checks if not p.exists()]

if missing:
    raise FileNotFoundError(
        'Missing saved model artifacts required for transfer evaluation. '
        'Run notebook 02_global_model_training_and_artifact_save.ipynb first. '
        f'Missing examples: {missing}'
    )

!python "{SCRIPT}" --m5-dir "{M5_DIR}" --granularity dept_store --datasets P --models XGBOOST CATBOOST --tag "{TAG}"


Saved predictions: /Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/modeling/outputs/reports/portable_m5_transfer_predictions.csv
Saved metrics: /Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/modeling/outputs/reports/portable_m5_transfer_metrics.csv
Saved summary: /Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/modeling/outputs/reports/portable_m5_transfer_summary.csv


## Summary

In [29]:
!python "/Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/modeling/scripts/m5_saved_artifact_transfer.py" \
  --m5-dir "/Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/external-data/m5-forecasting-accuracy" \
  --granularity dept_store \
  --datasets P \
  --models XGBOOST CATBOOST \
  --tag "portable_m5_transfer"


Saved predictions: /Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/modeling/outputs/reports/portable_m5_transfer_predictions.csv
Saved metrics: /Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/modeling/outputs/reports/portable_m5_transfer_metrics.csv
Saved summary: /Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/modeling/outputs/reports/portable_m5_transfer_summary.csv


In [30]:
summary_path = resolve_report_path('summary')
pd.read_csv(summary_path)


,dataset,model,split,horizon,n_obs,WAPE,RMSE,Bias,MASE_mean,wQL50,under_forecast_rate
0,M5_TRANSFER_P,XGBOOST,test,0,10080,0.256164,8089.558003,-3661.510512,3.475796,2168.748662,0.838492
1,M5_TRANSFER_P,CATBOOST,test,0,10080,0.370521,9995.585797,-6121.854851,5.128093,3136.929031,0.909425


## Detailed Horizon Metrics

In [31]:
metrics_path = resolve_report_path('metrics')
pd.read_csv(metrics_path).head(50)

,dataset,model,split,horizon,n_obs,WAPE,RMSE,Bias,MASE_mean,wQL50,under_forecast_rate
0,M5_TRANSFER_P,XGBOOST,test,1,840,0.210887,6011.105733,-3208.706199,3.095831,1785.426893,0.827381
1,M5_TRANSFER_P,XGBOOST,test,2,840,0.386832,13306.529924,-6387.876576,3.781956,3275.021944,0.866667
2,M5_TRANSFER_P,XGBOOST,test,3,840,0.242121,7972.049750,-3879.480738,3.252074,2049.862171,0.835714
3,M5_TRANSFER_P,XGBOOST,test,4,840,0.294838,8024.777553,-4752.642503,4.003201,2496.176352,0.870238
4,M5_TRANSFER_P,XGBOOST,test,5,840,0.398137,12715.827212,-6619.651531,4.210029,3370.727210,0.889286
5,M5_TRANSFER_P,XGBOOST,test,6,840,0.197270,6080.391358,-2183.303695,2.884384,1670.137874,0.795238
6,M5_TRANSFER_P,XGBOOST,test,7,840,0.193681,6042.219568,-2160.194854,2.801158,1639.750801,0.752381
7,M5_TRANSFER_P,XGBOOST,test,8,840,0.220099,6595.198432,-3300.792774,3.270795,1863.415718,0.848810
8,M5_TRANSFER_P,XGBOOST,test,9,840,0.208339,5941.803773,-3197.958929,3.126262,1763.851411,0.835714
9,M5_TRANSFER_P,XGBOOST,test,10,840,0.206047,5859.916668,-2496.327868,3.277416,1744.448114,0.807143


## Why This Is Not a Kaggle Submission

The saved artifacts are monthly synthetic-trained models. Kaggle M5 submission requires 28-day daily item-store forecasts. That means:
- this notebook is valid for transfer evaluation
- it is not valid for strict Kaggle submission generation from the same saved monthly artifacts